# 🎯 [K09_M03] 질문에 답하는 프롬프트 작성하기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K09/k09_m03_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·상담 (Level 2) / K09 부대 정보 검색 시스템 |
| **미션** | 미션 03. 질문에 답하는 프롬프트 작성하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 프롬프트 엔지니어링 및 팩트체크 검증 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k09_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K09", "../uploads/K09"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k09_m01~m10) 준비 완료!")



---

## 🎯 미션 03. 질문에 답하는 프롬프트 작성하기
**소요시간**: 약 1.5시간

![미션 3 외부 지식 주입 RAG 기본 프롬프트 설계 다이어그램](/uploads/common/K09_m03_context_injection.png)

### 📋 미션 개요 및 배경
AI 모델 자체 지식에 의존하지 않고 제공된 부대 안내 자료에만 근거하여 답변하도록 제한하는 기본 지시 프롬프트를 설계합니다. 앞서 정리한 자료표를 컨텍스트로 제공하고 우선 질문 5개를 입력하여, 자료가 포함되었을 때와 없을 때의 답변 정확도 차이를 실증 비교합니다. 이를 통해 검색 증강(RAG)의 기본 개념과 컨텍스트 기반 질의응답 원리를 이해합니다.

### 🎯 미션 학습목표
'제공된 안내 자료에만 근거하여 답하라'는 제약 지시문과 자료표를 프롬프트에 포함하여 질문 5개를 질의하고, 자료가 없을 때의 응답과 정확도를 실증 비교합니다. **이를 통해 외부 지식 주입(Context Injection)으로 AI의 답변 범위를 제어하고 근거 기반 답변을 생성하게 하는 검색 증강(RAG) 프롬프트 설계 원리를 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 자료 근거 지시와 자료표를 넣은 프롬프트를 만든다.
#### 이 단계의 역할
AI가 외부 지식이나 환각에 의존하지 않고 제공된 자료표 내에서만 답변을 도출하도록 제약하는 컨텍스트 주입 프롬프트를 설계합니다.

#### 해야 할 일

<li>시스템 역할(Role), 답변 원칙(Constraints), 컨텍스트(미션2 자료표)를 모듈화하여 프롬프트 뼈대를 작성합니다.

</li><li>"반드시 아래 [부대 안내 자료]에 근거하여 답하고 자료에 없는 내용은 절대 추측하지 말라"는 앵커링 문구를 배치합니다.

</li><li>AI 도구 채팅창에 지시문과 자료표를 주입(Context Injection)할 준비를 마칩니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[자료 기반 답변 기본 프롬프트 설계]:** "[역할]: 너는 한빛대대 부대 생활 정보 안내 도우미야.
[지시사항]: 아래 제공된 [부대 안내 자료]의 내용만을 근거로 장병의 질문에 답해줘. 자료에 명시되지 않은 사실은 절대로 지어내거나 추측하지 마.

[부대 안내 자료]:
{미션2 자료표 전문}

[질문]: 의무실 진료 시간과 물리치료실 이용 방법을 알려줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **제약 조건 명시:** '자료에 있는 내용만 근거로 답할 것'이 프롬프트 상단에 명시되었는가?
- **자료표 무결성:** 12개 시설의 정보가 프롬프트 컨텍스트 내에 온전히 포함되었는가?

#### 핵심 확인 지표
<blockquote>자료 주입형 기본 안내 프롬프트 서식

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>자료 기반 답변 프롬프트 초안

</li>

#### 🔹 Step 2. 우선 질문 5개를 한다.
#### 이 단계의 역할
작성된 자료 기반 프롬프트에 미션1에서 선별한 우선 질문 5개를 순차적으로 질의하여 AI의 답변을 수집합니다.

#### 해야 할 일

<li>우선 질문 1번부터 5번까지 차례대로 프롬프트에 넣어 질의를 실행합니다.

</li><li>AI가 생성한 각 질문별 응답 텍스트 전문을 추출하여 기록표에 정리합니다.

</li><li>답변에서 인용한 시설명, 운영 시간, 위치 정보가 자료표에 있는 내용인지 예비 확인합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[우선 질문 5건 연속 질의]:** "[프롬프트 본문]: {Step 1 프롬프트}
[질문 1]: 의무실 진료 시간과 야간 진료 접수 방법 알려줘.
[질문 2]: 평일 일과 후 PX(충성마트) 이용 시간은 언제인가요?
[질문 3]: 사이버지식정보방 위치와 주말 이용 시간은?
[질문 4]: 체력단련실 샤워장 이용 가능한 시간 알려줘.
[질문 5]: 병영도서관 도서 대출 권수와 기간은 어떻게 되나요?"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **5개 질문 전수 질의:** 선정된 우선 질문 5건이 빠짐없이 질의되었는가?
- **자료 인용 여부:** AI가 자신의 상식이 아닌 제공된 자료표의 내용을 인용하여 답변했는가?

#### 핵심 확인 지표
<blockquote>우선 질문 5개에 대한 AI 1차 응답 기록표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>우선 질문 5개 AI 질의 응답 로그

</li>

#### 🔹 Step 3. 자료 없이 같은 질문을 해 비교한다.
#### 이 단계의 역할
부대 안내 자료 없이 동일한 5개 질문을 AI에 질의하고, 자료가 있을 때와 없을 때의 응답을 대조하여 컨텍스트 주입의 효과를 실증 비교합니다.

#### 해야 할 일

<li>새 대화창을 열고 안내 자료표 없이 "한빛대대 의무실 운영 시간을 알려줘" 등 동일 5개 질문을 질의합니다.

</li><li>자료가 없을 때 AI가 생성하는 막연한 일반론, 추측성 답변 및 허위 정보(환각)를 기록합니다.

</li><li>[자료 주입 답변 vs 자료 미제공 답변]의 구체성, 정확도, 환각 여부를 비교표로 정리합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[자료 미제공 비교 질의]:** "한빛대대 의무실 운영 시간과 위치, 야간 진료 접수 방법을 알려줘. (안내 자료표 주입 없이 단독 질문)"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **환각 현상 식별:** 자료가 없을 때 AI가 실제와 다른 가상의 시간이나 부서명을 지어냈는지 식별했는가?
- **비교표 완성도:** 5개 질문 각각에 대해 자료 유/무에 따른 답변 차이가 구체적으로 대조되었는가?

#### 핵심 확인 지표
<blockquote>자료 유무에 따른 AI 응답 비교표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>자료 유무 비교표(필수 산출물)

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
자료표에서 특정 시설 정보를 일부 삭제한 부분 자료표를 주입하여, 부분적 정보 부재 시 AI의 반응을 스트레스 테스트합니다.

#### 해야 할 일
자료표 일부만 넣었을 때 어떤 답이 나오는지 확인해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[부분 결손 자료표 반응 시험 질의]:** "[부분 자료표]: {체력단련실 정보가 삭제된 자료표}
[질문]: 체력단련실 헬스장 샤워실 운영 시간 알려줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **정보 경계 인지:** 삭제된 시설에 대해 AI가 지어내지 않고 정보가 없음을 올바르게 인지하는가?

#### 핵심 확인 지표
<blockquote>부분 결손 조건 스트레스 테스트 결과 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 도전 과제 수행 결과 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M03] 질문에 답하는 프롬프트 작성하기 실습 가이드 및 작성 서식

## 1. 자료 기반 답변 프롬프트 서식 (Context Injection)

```markdown
[역할]: 너는 육군 가상 한빛대대 부대 생활 정보 질의응답 도우미야.
[지시사항]:
1. 반드시 아래 [부대 생활 안내 자료]에 명시된 사실에만 근거하여 장병의 질문에 답변하라.
2. 자료에 명시되지 않은 내용은 절대로 임의로 추측하거나 상상하여 지어내지 말라.
3. 친절하고 신뢰감 있는 군 경어체(합니다/습니다 체)로 답변하라.

[부대 생활 안내 자료]:
{미션2에서 정리한 12건 안내 자료표 전문}

[사용자 질문]:
{질문 입력}
```

---

## 2. 자료 유/무 비교 대조표 (5개 질문 실증 테스트)

| 질문 번호 | 질문 내용 | 자료 주입 시 AI 응답 (정확도) | 자료 미제공 시 AI 응답 (환각 발생 여부) | 분석 결과 |
| :---: | :--- | :--- | :--- | :--- |
| **TQ01** | 의무실 진료 시간 및 야간 접수 | 평일 09:00~17:30(소독 12~13:30 제외), 야간 당직 군의관 접수 (정확) | "통상 군 부대 의무실은 09시부터 17시까지..." 모호한 추측 및 가상 번호 날조 | 자료 주입 시 100% 팩트 일치 |
| **TQ02** | 평일 일과 후 PX 이용 시간 | 평일 12:00~19:30, 14~15시 재고조사 제외 (정확) | "일과 후 18:00~20:00까지 운영됩니다" 오답 생성 | 자료표의 예외 시간 정확 반영 |
| **TQ03** | 사지방 위치 및 주말 이용 시간 | 복지회관 2층 201호, 주말 09:00~21:00 (정확) | 위치를 본관 3층 등으로 허위 생성 | 건물 번호 및 호수까지 완벽 매칭 |
| **TQ04** | 체단실 이용 시간 및 방역 시간 | 평일 06:30~21:00, 수요일 14~16시 방역 (정확) | 방역 소독 시간 완전 누락 | 예외 조항 누락 방지 효과 확인 |
| **TQ05** | 도서관 대출 권수 및 기간 | 1인 3권, 14일간 대출 가능 (정확) | "보통 2권 7일간 대출 가능합니다" 일반론 왜곡 | 정확한 대출 규정 인용 확인 |

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **부대 생활 안내 자료 주입형 기본 프롬프트 템플릿 (MD):** 시스템 역할(Role), 자료 컨텍스트(Context), 제약조건(Constraints) 모듈형 지시문
- **우선 질문 5선 테스트 시트 및 질의응답 기록부:** 5개 핵심 질문별 질의 실행 및 AI 원시 응답 추출 양식
- **[학습 가이드] 외부 지식 주입(Context Injection)과 RAG 기초 원리:** 모델 자체 지식이 아닌 제공된 데이터셋에만 근거하여 답을 생성하게 제한하는 프롬프트 엔지니어링 원리

**실습 팁:** 프롬프트 상단에 "아래 [부대 안내 자료]에 없는 내용은 절대 추측하여 지어내지 말 것"이라는 강력한 앵커링 문구를 배치해야 환각을 예방할 수 있습니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 프롬프트 작성(자료 근거) — 자료에만 근거하도록 제약 지시문을 설계하고 자료 유/무에 따른 답변 차이를 실증 비교했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>자료 주입형 기본 안내 프롬프트 서식 완성

</li><li>우선 질문 5건 전수 질의 실행 및 로그 수집

</li><li>자료 유/무에 따른 정확도 및 환각(Hallucination) 비교표 완성

</li>

#### 채점자 판정 메모
<blockquote>자료가 없을 때 AI가 임의로 지어낸 답변(예: 가상 부대 시간 왜곡)을 명확히 식별하고, 자료 주입의 효과를 입증했는지 확인한다.

</blockquote>





In [ ]:
# [미션 03. 질문에 답하는 프롬프트 작성하기] 학생 실습 프롬프트 및 결과 기록란
mission_03_prompt = """
# [TODO: 미션 03] Context Injection(자료 주입) 기법을 적용하여 12건의 자료표만을 근거로 답변하도록 시스템 프롬프트를 작성하세요.
# 힌트: 역할, 엄격한 근거 준수 지시, 12건 자료표, 그리고 5대 우선 질문을 순차적으로 포함합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_03_result = """
### 📋 [미션 03 실습 결과]
## 1. 자료 기반 질의응답 실증 테스트 (5개 질문)
| 질문 번호 | 질문 내용 | 자료 주입 시 AI 응답 | 자료 미제공 시 응답 (환각 여부) | 분석 결과 |
| :---: | :--- | :--- | :--- | :--- |
| **TQ01** | 의무실 진료 시간 및 야간 접수 | | | |
| **TQ02** | 사지방 주말 운영 시간 | | | |
| **TQ03** | 체단실 주말 샤워 온수 공급 | | | |
| **TQ04** | PX 평일 운영 시간 | | | |
| **TQ05** | 도서관 책 대출 권수 및 반납 | | | |
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 03. 질문에 답하는 프롬프트 작성하기", mission_03_prompt, mission_03_result)

